In [7]:
import sys

sys.path.append("../scripts")

In [8]:
import pandas as pd
import numpy as np
import preprocessing

In [9]:
data = pd.read_csv('../data/car_price_prediction.csv')
data.head()

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
0,45654403,13328,1399,LEXUS,RX 450,2010,Jeep,Yes,Hybrid,3.5,186005 km,6.0,Automatic,4x4,04-May,Left wheel,Silver,12
1,44731507,16621,1018,CHEVROLET,Equinox,2011,Jeep,No,Petrol,3,192000 km,6.0,Tiptronic,4x4,04-May,Left wheel,Black,8
2,45774419,8467,-,HONDA,FIT,2006,Hatchback,No,Petrol,1.3,200000 km,4.0,Variator,Front,04-May,Right-hand drive,Black,2
3,45769185,3607,862,FORD,Escape,2011,Jeep,Yes,Hybrid,2.5,168966 km,4.0,Automatic,4x4,04-May,Left wheel,White,0
4,45809263,11726,446,HONDA,FIT,2014,Hatchback,Yes,Petrol,1.3,91901 km,4.0,Automatic,Front,04-May,Left wheel,Silver,4


In [10]:
data = data.astype('object')
data = preprocessing.preprocessing_pipeline(data)

Preprocessing started...
initial shape: (19237, 18)
After dropping duplicates: (18924, 18)
Replacing categorical values with numerical...
After cleaning outliers: (16312, 18)
Feature engineering...
Droping unnecessary columns...
Final shape after preprocessing: (16312, 16)


In [11]:
data.head()

,Price,Levy,Manufacturer,Model,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Wheel,Color,Airbags,Age
0,13328,1399,LEXUS,RX 450,Jeep,Yes,Hybrid,3.5,186005,6.0,Automatic,4x4,Left wheel,Silver,12,16
1,16621,1018,CHEVROLET,Equinox,Jeep,No,Petrol,3.0,192000,6.0,Tiptronic,4x4,Left wheel,Black,8,15
2,8467,0,HONDA,FIT,Hatchback,No,Petrol,1.3,200000,4.0,Variator,Front,Right-hand drive,Black,2,20
3,3607,862,FORD,Escape,Jeep,Yes,Hybrid,2.5,168966,4.0,Automatic,4x4,Left wheel,White,0,15
4,11726,446,HONDA,FIT,Hatchback,Yes,Petrol,1.3,91901,4.0,Automatic,Front,Left wheel,Silver,4,12


In [12]:
from sklearn.preprocessing import LabelEncoder,StandardScaler,OneHotEncoder

one_hot_columns = ['Leather interior', 'Gear box type', 'Drive wheels', 'Wheel']

# data = pd.get_dummies(data, columns=one_hot_columns)
oh_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
oh_encoded_train = oh_encoder.fit_transform(data[one_hot_columns])

oh_encoded_columns = oh_encoder.get_feature_names_out(one_hot_columns)

In [13]:
oh_encoded_train_df = pd.DataFrame(oh_encoded_train, columns=oh_encoded_columns, index=data.index)

In [14]:
data = pd.concat([data, oh_encoded_train_df], axis=1)
data.drop(columns=one_hot_columns, inplace=True)

In [15]:
import pickle
#save the one hot encoder for later use
with open('../models/one_hot_encoder.pkl', 'wb') as f:
    pickle.dump(oh_encoder, f)

In [16]:
label_encode_columns = ['Fuel type', 'Category', 'Model', 'Color', 'Manufacturer']

label_encoders = {}
for column in label_encode_columns:
    label_encoder = LabelEncoder()
    data[column] = label_encoder.fit_transform(data[column])
    label_encoders[column] = label_encoder

In [17]:
# Save the label encoders for later use
with open('../models/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

In [18]:
X = data.drop('Price', axis=1)
y = data['Price']

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

print(f'Train set: {len(X_train)} samples')
print(f'Test set: {len(X_test)} samples')

Train set: 13865 samples
Test set: 2447 samples


In [20]:
numerical_columns = ['Levy', 'Engine volume', 'Mileage', 'Age']

scaler = StandardScaler()
X_train[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

In [21]:
#save Scaler for later use
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [22]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

rf = RandomForestRegressor()
rf.fit(X_train, y_train)

y_test_pred = rf.predict(X_test)
rmse = root_mean_squared_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print(f'Validation RMSE: {rmse}')
print(f'Validation R2: {r2}')

Validation RMSE: 5607.7579605539695
Validation R2: 0.7611061617276635


In [23]:
#save Random Forest model for later use
with open('../models/model.pkl', 'wb') as f:
    pickle.dump(rf, f)